# From Space to Action - Agricultural Drought Early Warning
## Notebook 04: Target Construction
**Goal:** Define drought classes and create prediction targets.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/FromSpaceToAction'
DATA_DIR = f'{PROJECT_DIR}/data'
SRC_DIR = f'{PROJECT_DIR}/src'
MODELS_DIR = f'{PROJECT_DIR}/models'
OUTPUTS_DIR = f'{PROJECT_DIR}/outputs'
CONFIG_PATH = f'{PROJECT_DIR}/config/config.yaml'

sys.path.insert(0, SRC_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


### Load feature data

In [ ]:
features_path = f'{DATA_DIR}/features/features_v1.parquet'
if os.path.exists(features_path):
    df = pd.read_parquet(features_path)
    print(f"Loaded feature data shape: {df.shape}")
else:
    print("Feature data not found!")

### Section: VCI-based drought classification

In [ ]:
def categorize_drought(vci):
    if vci < 20: return 'Severe'
    if vci < 35: return 'Warning'
    if vci <= 40: return 'Watch'
    return 'Normal'

df['drought_class'] = df['vci'].apply(categorize_drought)
print("Drought classes assigned.")

### Section: Class distribution analysis

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='drought_class', order=['Normal', 'Watch', 'Warning', 'Severe'])
plt.title('Drought Class Distribution')
plt.show()

### Section: Lead time target construction

In [ ]:
df = df.sort_values(by=['lat', 'lon', 'dekad'])

df['target_lead1'] = df.groupby(['lat', 'lon'])['drought_class'].shift(-1)
df['target_lead2'] = df.groupby(['lat', 'lon'])['drought_class'].shift(-2)
df['target_lead3'] = df.groupby(['lat', 'lon'])['drought_class'].shift(-3)

df = df.dropna(subset=['target_lead1', 'target_lead2', 'target_lead3'])
print("Lead time targets constructed.")

### Section: Leakage audit

In [ ]:
print("Leakage Audit: Targets are shifted backwards; past data features correspond to future target class. No future info in current row features.")

### Section: Train/Val/Test split

In [ ]:
df['year'] = df['dekad'].dt.year
df['month'] = df['dekad'].dt.month

def assign_split(row):
    if row['year'] <= 2021: return 'Train'
    elif row['year'] == 2022 or (row['year'] == 2023 and row['month'] <= 6): return 'Val'
    else: return 'Test'

df['split'] = df.apply(assign_split, axis=1)

print("Split sizes:")
print(df['split'].value_counts())
print("\nClass distribution per split:")
print(df.groupby(['split', 'target_lead1']).size())

### Save final dataset

In [ ]:
targets_path = f'{DATA_DIR}/targets/dataset_v1.parquet'
df.to_parquet(targets_path)
print(f"Final dataset saved to {targets_path}")